# El Oráculo inmobiliario de California

En este notebook se van a entrenar 2 modelos predictivos básicos de Machine Learning y consta de misiones:

**Misión 1:** Predecir el Precio del Sueño Californiano.

Predecir el precio de las casas de California basándonos en el *dataset* `housing.csv`, utilizando *regresión lineal*.

**Misión 2:** Detectar al Cliente Traidor.

Predecir qué clientes se van, utilizando *regresión logística*.

## Misión 1: Predecir el Precio del Sueño Californiano

### Recordatorio del EDA

In [42]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import root_mean_squared_error, confusion_matrix, accuracy_score, precision_score, recall_score


df = pd.read_csv("housing.csv")
df.shape

(20640, 10)

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


In [3]:
df.corr(numeric_only=True)["median_house_value"].sort_values(ascending=False)

median_house_value    1.000000
median_income         0.688075
total_rooms           0.134153
housing_median_age    0.105623
households            0.065843
total_bedrooms        0.049686
population           -0.024650
longitude            -0.045967
latitude             -0.144160
Name: median_house_value, dtype: float64

In [4]:
df.groupby("ocean_proximity")["median_house_value"].median()

ocean_proximity
<1H OCEAN     214850.0
INLAND        108500.0
ISLAND        414700.0
NEAR BAY      233800.0
NEAR OCEAN    229450.0
Name: median_house_value, dtype: float64

In [5]:
df["ocean_proximity"].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

In [6]:
df["median_house_value"].value_counts().head()

median_house_value
500001.0    965
137500.0    122
162500.0    117
112500.0    103
187500.0     93
Name: count, dtype: int64

In [7]:
df["housing_median_age"].value_counts().head()

housing_median_age
52.0    1273
36.0     862
35.0     824
16.0     771
17.0     698
Name: count, dtype: int64

In [8]:
df["median_income"].value_counts().head()

median_income
3.1250     49
15.0001    49
2.8750     46
2.6250     44
4.1250     44
Name: count, dtype: int64

In [9]:
df["median_income"].max()

np.float64(15.0001)

En el análisis anterior del dataset encontramos varios hallazgos importantes. Los números de abajo salen del dataset completo, antes de limpiarlo.

- **Ingreso mediano:** La variable que más se asocia con el precio de las zonas es `median_income`, con r = 0,688.
- **Ubicación:** El precio mediano más bajo es el de las zonas `INLAND` (108.500) y el más alto es el de `ISLAND` (414.700), aunque `ISLAND` tiene solo 5 zonas.
- **Topes:** hay valores que se repiten de forma sospechosa: 500.001 en `median_house_value` (965 zonas), 52 en `housing_median_age` (1.273 zonas) y 15,0001 en `median_income` (49 zonas), que parece un tope.
- **Nulos:** `total_bedrooms` tiene 207 zonas con valores nulos.


### Preparación de datos

Se decidió **conservar** las 207 zonas con valores nulos en `total_bedrooms`. Esa columna tiene una correlación de 0,0497 con el precio `median_house_value`, mientras que esas zonas tienen completo `median_income`, la variable con la correlación más alta con el precio que es 0,688. Los nulos se rellenarán después de la división train/test.

In [10]:
pd.get_dummies(df["ocean_proximity"]).head()

,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
0,False,False,False,True,False
1,False,False,False,True,False
2,False,False,False,True,False
3,False,False,False,True,False
4,False,False,False,True,False


In [11]:
ubicacion = pd.get_dummies(df["ocean_proximity"]).drop(columns=["INLAND"])
ubicacion.sum()

<1H OCEAN     9136
ISLAND           5
NEAR BAY      2290
NEAR OCEAN    2658
dtype: int64

Ya que `ocean_proximity` es una columna de categorías, el modelo de regresión lineal no puede usarla directamente, entonces cada una de sus 5 categorías se convirtió en su propia columna booleana, que el modelo sí puede utilizar.

Se eliminó la columna `INLAND`, que es la categoría con la mediana de precio más baja, 108.500. Sus zonas, 6.551, se reconocen porque las otras cuatro columnas valen `False`. Se quita la columna porque con las cinco, una es redundante y los coeficientes no tendrían un valor único. Con `INLAND` como referencia, el coeficiente de cada categoría se lee respecto a ella.

In [12]:
df_california = pd.concat([df.drop(columns=["ocean_proximity"]), ubicacion], axis=1)
df_california.shape

(20640, 13)

El DataFrame `df_california` conserva las 20.640 zonas del DataFrame original y ya no tiene columnas de texto: se quitó `ocean_proximity` y se agregaron las 4 columnas de `ubicacion`, por lo que ahora tiene 13 columnas en total. Todavía quedan las 207 zonas con `total_bedrooms` nulo, que se rellenan después de la división train/test.

### División train/test

In [13]:
X = df_california.drop(columns=["median_house_value"])
y = df_california["median_house_value"]
X.shape, y.shape

((20640, 12), (20640,))

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=36)
X_train.shape, X_test.shape

((16512, 12), (4128, 12))

Se separó `df_california` en `X`, con las 12 variables para predecir, e `y`, con `median_house_value`, que es lo que el modelo tiene que predecir. El precio se sacó de `X` para que el modelo no lo use como una pista. Después, `X` e `y` se dividieron en entrenamiento y test.

Se tomó la decisión de dividir 80/20 para que el modelo tenga una buena cantidad de datos para entrenar, 16.512 zonas, y una buena cantidad para la prueba, 4.128. Se decidió el valor `36` para `random_state`, ya que conviene que la división sea siempre la misma y los resultados se puedan repetir.

### Relleno de nulos

In [15]:
X_train["total_bedrooms"].isnull().sum(), X_test["total_bedrooms"].isnull().sum()

(np.int64(161), np.int64(46))

In [16]:
X_train["total_bedrooms"].median()

np.float64(434.0)

In [17]:
mediana_bedrooms = X_train["total_bedrooms"].median()
X_train["total_bedrooms"] = X_train["total_bedrooms"].fillna(mediana_bedrooms)
X_test["total_bedrooms"] = X_test["total_bedrooms"].fillna(mediana_bedrooms)
X_train["total_bedrooms"].isnull().sum(), X_test["total_bedrooms"].isnull().sum()

(np.int64(0), np.int64(0))

Se tomó la decisión de completar los nulos de `X_train` y `X_test` con la mediana de `X_train`, que es 434, para evitar el data leakage de la información del test en el entrenamiento. Se realizó con la mediana y no con el promedio, porque este último puede tener valores extremos. Se encontraron que habían 161 datos nulos en `X_train` y 46 nulos en `X_test`, todos fueron rellenados satisfactoriamente.

### Entrenamiento

In [18]:
modelo_california = LinearRegression()
modelo_california.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](12,)","[-26951.23,-25593.44, 1044.17,...,196011.77, 35799.91, 45280.14]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](12,)","['longitude','latitude','housing_median_age',...,'ISLAND','NEAR BAY', 'NEAR OCEAN']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-2.319e+06
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,12
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(12)


Se creó el `modelo_california` con `LinearRegression` y se lo entrenó con los datos de `X_train` e `y_train`. Aprendió 12 coeficientes, uno por cada columna, y un intercepto.

### Evaluación

In [19]:
y_pred = modelo_california.predict(X_test)
y_pred.shape, y_pred[:5]

((4128,),
 array([172760.46886155,  85296.31938491, 289907.64112096, 183109.37234378,
        269229.95166455]))

In [20]:
y_test.head()

13210    182600.0
12777     60500.0
5288     500001.0
5632     186800.0
17603    274100.0
Name: median_house_value, dtype: float64

In [21]:
rmse = root_mean_squared_error(y_test, y_pred)
rmse

68754.18432254338

`modelo_california` predijo los precios de las 4.128 zonas del test, que no había visto durante el entrenamiento. El RMSE fue de 68.754: en promedio, el modelo se equivoca unos 68.754 dólares, y los errores grandes pesan más en este número.

### Interpretación de los coeficientes

In [22]:
coeficientes = pd.Series(modelo_california.coef_, index=X_train.columns)
coeficientes.sort_values(ascending=False)

ISLAND                196011.768278
NEAR OCEAN             45280.136972
<1H OCEAN              39472.659958
median_income          38653.633468
NEAR BAY               35799.908649
housing_median_age      1044.173254
households                73.861526
total_bedrooms            71.004728
total_rooms               -4.511818
population               -38.188934
latitude              -25593.438380
longitude             -26951.231039
dtype: float64

Los coeficientes de `modelo_california` se leen manteniendo las demás variables iguales. Por ejemplo, cada unidad más de `median_income` se asocia con unos 38.654 más de precio predicho. Son asociaciones y no causas, y no se puede comparar su tamaño entre variables, porque cada coeficiente está medido en la unidad de su variable, cantidades, ingresos, edades, etc.

`INLAND` es la referencia de las columnas de ubicación. Una zona `NEAR OCEAN` tiene un precio predicho 45.280 más alto que una `INLAND`, una `<1H OCEAN` 39.473 más y una `NEAR BAY` 35.800 más. Por otro lado `ISLAND` tiene el coeficiente más alto, 196.012, pero son solo 5 zonas, por eso el valor es poco fiable.

## Misión 2: Detectar al Cliente Traidor

### Exploración del dataset

In [23]:
df_churn = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df_churn.shape, df_churn["Churn"].value_counts()

((7043, 21),
 Churn
 No     5174
 Yes    1869
 Name: count, dtype: int64)

In [24]:
df_churn.info()
df_churn.head()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [25]:
df_churn["TotalCharges"].value_counts().head()

TotalCharges
20.2     11
         11
19.75     9
19.9      8
20.05     8
Name: count, dtype: int64

El dataset tiene 7.043 clientes y 21 columnas. En la columna `Churn` `Yes` indica si el cliente se fue, 1.869 clientes, y `No`, 5.174 indica que se quedó, así que la mayoría de los clientes se queda.

### Preparación de datos

In [26]:
for columna in ["Contract", "InternetService", "PaperlessBilling"]:
    print(df_churn[columna].value_counts())

Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64
PaperlessBilling
Yes    4171
No     2872
Name: count, dtype: int64


In [27]:
df_churn = df_churn[["tenure", "MonthlyCharges", "SeniorCitizen", "Contract", "InternetService", "PaperlessBilling", "Churn"]]
df_churn.shape

(7043, 7)

Para simplificar el dataset se conservaron 6 columnas más `Churn`: Tres numéricas, `tenure`, `MonthlyCharges` y `SeniorCitizen`. Tres de texto con pocas categorías `Contract`, `InternetService` y `PaperlessBilling`. Se descartó `customerID` porque es un identificador, y `TotalCharges` porque figura como texto y tiene 11 valores vacíos. Las demás columnas no se usaron para mantener el modelo simple.

In [29]:
y_churn = (df_churn["Churn"] == "Yes").astype(int)
X_churn = df_churn.drop(columns=["Churn"])
y_churn.value_counts(), X_churn.shape

(Churn
 0    5174
 1    1869
 Name: count, dtype: int64,
 (7043, 6))

Se separó la columnas `Churn` como `y_churn`, el cliente que se fue ahora vale 1 y el que se quedó vale 0, las demás columnas quedaron como `X_churn` con 5.174 filas e `y_churn` quedo con 1.869.

In [30]:
X_churn = pd.get_dummies(X_churn, drop_first=True)
X_churn.shape, X_churn.columns

((7043, 8),
 Index(['tenure', 'MonthlyCharges', 'SeniorCitizen', 'Contract_One year',
        'Contract_Two year', 'InternetService_Fiber optic',
        'InternetService_No', 'PaperlessBilling_Yes'],
       dtype='str'))

Las columnas de texto de `X_churn` se convirtieron en columnas booleanas, quitando la primera categoría de cada una como referencia, `Month-to-month`, `DSL` y `No`. Así `X_churn` pasó de 6 a 8 columnas, y los coeficientes de cada categoría se van a leer respecto a su referencia.

### División train/test

In [31]:
X_train_churn, X_test_churn, y_train_churn, y_test_churn = train_test_split(X_churn, y_churn, test_size=0.2, random_state=36)
X_train_churn.shape, X_test_churn.shape, y_train_churn.value_counts()

((5634, 8),
 (1409, 8),
 Churn
 0    4170
 1    1464
 Name: count, dtype: int64)

`X_churn` e `y_churn` se dividieron en entrenamiento y prueba con una proporción 80/20 y `random_state=36`, igual que en la Misión 1. El entrenamiento tiene 5.634 clientes y el test 1.409, con 8 columnas cada uno. En el entrenamiento hay 4.170 clientes con 0 y 1.464 con 1.

### Entrenamiento

In [34]:
modelo_churn = LogisticRegression(max_iter=1000)
modelo_churn.fit(X_train_churn, y_train_churn)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

Se creó `modelo_churn` con `LogisticRegression` y se entrenó con los datos de `X_train_churn` e `y_train_churn`.

In [35]:
y_pred_churn = modelo_churn.predict(X_test_churn)
y_pred_churn.shape, y_pred_churn[:5], y_test_churn.head()

((1409,),
 array([0, 0, 0, 0, 0]),
 2661    0
 2836    0
 786     0
 6556    0
 4436    0
 Name: Churn, dtype: int64)

`modelo_churn` predijo que para los 1.409 clientes del test, si se van (1) o se quedan (0).
Los primeros 5 valores predichos son 0, igual que los reales de esos clientes.

In [37]:
confusion_matrix(y_test_churn, y_pred_churn)

array([[893, 111],
       [195, 210]])

In [39]:
pd.DataFrame(
    confusion_matrix(y_test_churn, y_pred_churn),
    index=["Real: se quedó", "Real: se fue"],
    columns=["Predijo: se queda", "Predijo: se va"],
)

,Predijo: se queda,Predijo: se va
Real: se quedó,893,111
Real: se fue,195,210


En los 1.409 clientes del test, `modelo_churn` acertó en su predicción cuando dijo que 893 clientes se quedaban y cuando dijo que 210 se iban. Se equivocó en su predicción cuando dijo que 111 se iban, pero se quedaron, y cuando dijo que 195 se quedaban, pero eran falsos negativos y se fueron.

In [41]:
accuracy_score(y_test_churn, y_pred_churn)

0.7828246983676366

In [43]:
recall_score(y_test_churn, y_pred_churn)

0.5185185185185185

In [44]:
precision_score(y_test_churn, y_pred_churn)

0.6542056074766355

In [45]:
print("Accuracy:", accuracy_score(y_test_churn, y_pred_churn))
print("Precision:", precision_score(y_test_churn, y_pred_churn))
print("Recall:", recall_score(y_test_churn, y_pred_churn))
y_test_churn.value_counts()

Accuracy: 0.7828246983676366
Precision: 0.6542056074766355
Recall: 0.5185185185185185


Churn
0    1004
1     405
Name: count, dtype: int64

Las métricas del `modelo_churn` en el test son:
- Accuracy: 0,78
- Precision: 0,65
- Recall: 0,52  

El accuracy parece alto, pero hay que leerlo con cuidado: de los 1.409 clientes del test, 1.004 se quedaron realmente, así que un modelo que dijera siempre "se queda" acertaría en esos 1.004 casos sin detectar a ningún cliente que se va. El recall indica que el modelo detecta poco más de la mitad de los clientes que se van, 210 de 405, y la precisión indica que, cuando dice que un cliente se va, acierta en unos dos de cada tres casos 210 de 321.

### Interpretación de los coeficientes

In [46]:
coeficientes_churn = pd.Series(modelo_churn.coef_[0], index=X_train_churn.columns)
coeficientes_churn.sort_values(ascending=False)

InternetService_Fiber optic    0.930779
PaperlessBilling_Yes           0.452716
SeniorCitizen                  0.365908
MonthlyCharges                 0.002706
tenure                        -0.034070
Contract_One year             -0.748536
InternetService_No            -0.891062
Contract_Two year             -1.605031
dtype: float64

Los coeficientes de `modelo_churn` se leen con las demás variables iguales: un valor positivo aumenta la probabilidad de que el cliente se vaya, y uno negativo la disminuye. Respecto a sus referencias, los clientes con fibra óptica 0,93 tienen más probabilidad de irse que los de `DSL`, y los de contrato de uno o dos años tienen menos que los de contrato mensual, con -0,75 y -1,61 respectivamente. `tenure` tiene un coeficiente negativo, -0,03: más antigüedad se asocia con menos probabilidad de irse. Son asociaciones y no causas, y no se compara el tamaño entre variables con unidades distintas.